# Project - Arabic Sentiment Analysis **(LAB)** 👨🏻‍💻📖

## 1️⃣ Required Libraries

In [2]:
# Basic Libraries
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import re
import os
import seaborn as sns

# NLP Libraries
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer

# Word Cloud
from wordcloud import WordCloud
from collections import Counter

# Modeling
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 2️⃣ Read Dataset

### Read From Local File

In [4]:
import pandas as pd
df = pd.read_excel(r'C:\Users\maged\Downloads\nlp\reviews.xlsx')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\maged\\Downloads\\nlp\\reviews.xlsx'

In [ ]:
df.shape

In [ ]:
df.head()

#### 📂 **Project Data Source: Arabic Reviews Dataset**

##### 1. Overview
The **Arabic Reviews Dataset** is a labeled collection of Arabic-language user reviews designed for **binary sentiment classification** tasks.

* **Platform:** Kaggle.
* **Task:** Binary Sentiment Classification (Positive / Negative).
* **Our Focus:** Arabic text preprocessing + Naive Bayes classification.

---

##### 2. Data Content & Structure

| Column | Description | Example |
| :--- | :--- | :--- |
| **text** | Raw Arabic review text | هذا الفيلم رائع جداً |
| **label** | Sentiment class | Positive / Negative |

---

##### 3. Technical Specifications
After loading the raw data for this notebook, the dataset characteristics are as follows:

* **Language:** Arabic (`ar`)
* **Classes:** Positive, Negative
* **Format:** CSV (comma-separated values)
* **Task Type:** Binary Text Classification

---

##### 4. Why This Dataset for this Project?
1.  **Real-World Arabic Text:** Reviews contain natural Arabic with dialects, punctuation, and noise — perfect for practicing preprocessing.
2.  **Balanced Task:** Binary classification makes it ideal for applying Naive Bayes from scratch.
3.  **NLP-Friendly Size:** Small enough to run without GPUs, large enough to produce meaningful results.


## 3️⃣ Exploratory Data Analysis (EDA)

### Information

In [ ]:
df.info()

### Check Columns

In [ ]:
df.columns

### Description

In [ ]:
df.describe()

### Check Duplications

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

### Check Missing Values

In [ ]:
df.isna().sum()

### Show Number of Unique Values

In [ ]:
for col in df.columns:
    print('{} : {} unique value(s)'.format(col, df[col].nunique()))

### Label Distribution

In [ ]:
print(df['label'].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
counts = df['label'].value_counts()
colors = ['steelblue', 'tomato']

bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white', width=0.5)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 50,
            str(int(bar.get_height())),
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Label Distribution (Positive vs Negative)', fontsize=14, pad=15)
ax.set_xlabel('Sentiment Label')
ax.set_ylabel('Number of Reviews')
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout()
plt.show()

### Show Most Common Reviews

In [ ]:
top_reviews = df['text'].value_counts().head(15).reset_index()
top_reviews.columns = ['Review', 'Count']
top_reviews.style.background_gradient(cmap='Greens')

### Make Column: Length of Review

In [ ]:
df['number_of_characters'] = df['text'].str.len()
df['number_of_words'] = df['text'].apply(lambda x: len(str(x).split()))

df.head()

In [ ]:
print(f"Maximum number of Characters in a review: {df['number_of_characters'].max()}")
print(f"Maximum number of Words in a review: {df['number_of_words'].max()}")

#### Distribution of Review Lengths

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
colors = ['steelblue', 'tomato']

axes[0].hist(df['number_of_characters'], bins=40, color=colors[0], edgecolor='white', range=(0, 400))
axes[0].set_title('Distribution of Review Length (Characters)', fontsize=12)
axes[0].set_xlabel('Length (Characters)')
axes[0].set_ylabel('Number of Reviews')
axes[0].set_xticks(np.arange(0, 401, 40))
axes[0].set_xlim(0, 400)

axes[1].hist(df['number_of_words'], bins=40, color=colors[1], edgecolor='white', range=(0, 80))
axes[1].set_title('Distribution of Review Length (Words)', fontsize=12)
axes[1].set_xlabel('Length (Words)')
axes[1].set_ylabel('Number of Reviews')
axes[1].set_xticks(np.arange(0, 81, 10))
axes[1].set_xlim(0, 80)

plt.tight_layout()
plt.show()

#### Generating Word Cloud For Reviews

In [ ]:
all_words = " ".join(df['text'].astype(str)).split()
word_counts = Counter(all_words)

wordcloud = WordCloud(
    font_path=None,
    max_font_size=50,
    max_words=200,
    background_color='white'
).generate_from_frequencies(word_counts)

plt.figure(figsize=(14, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Most Common Words in Arabic Reviews', fontsize=14)
plt.show()

## 4️⃣ Processing

### Clean Texts

In [ ]:
def clean_text(text):
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove punctuation, numbers, and special characters (keep Arabic letters and spaces)
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    # Remove extra whitespace
    text = " ".join(text.split())
    return text

### Arabic Normalization

In [ ]:
def normalize_arabic(text):
    # Normalize alef variants
    text = re.sub("[إأآ]", "ا", text)
    # Normalize ya variants
    text = re.sub("ى", "ي", text)
    # Normalize ta marbuta
    text = re.sub("ة", "ه", text)
    # Remove diacritics (tashkeel)
    text = re.sub(r'[\u064B-\u0652]', '', text)
    return text

### Tokenization

In [ ]:
def tokenize_arabic(text):
    return text.split()

### Stopword Removal

In [ ]:
arabic_stopwords = set(stopwords.words('arabic'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in arabic_stopwords]

### Stemming

In [ ]:
stemmer = ISRIStemmer()

def stem_arabic(tokens):
    return [stemmer.stem(word) for word in tokens]

### Apply All Preprocessing Steps

In [ ]:
def full_preprocess(text):
    text = clean_text(str(text))
    text = normalize_arabic(text)
    tokens = tokenize_arabic(text)
    tokens = remove_stopwords(tokens)
    tokens = stem_arabic(tokens)
    return " ".join(tokens)

df['processed_text'] = df['text'].apply(full_preprocess)

df[['text', 'processed_text']].head()

### Convert Text to Features (Bag of Words)

In [ ]:
vectorizer = CountVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['processed_text'])
y = df['label']

In [ ]:
print(f"Feature Matrix Shape: {X.shape}")
print(f"Vocabulary Size: {len(vectorizer.vocabulary_)}")
print(f"Label Distribution:\n{y.value_counts()}")

### Splitting Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(f"Training samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")

## 5️⃣ Modeling

### Train Naive Bayes

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)

### Evaluation

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

In [ ]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
labels = model.classes_

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, linecolor='gray')
plt.title('Confusion Matrix', fontsize=14, pad=15)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

#### Plot Evaluation Metrics

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

metrics = {
    'Accuracy' : accuracy_score(y_test, y_pred) * 100,
    'Precision': precision_score(y_test, y_pred, pos_label='Positive') * 100,
    'Recall'   : recall_score(y_test, y_pred, pos_label='Positive') * 100,
    'F1-Score' : f1_score(y_test, y_pred, pos_label='Positive') * 100,
}

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")
palette = sns.color_palette("husl", len(metrics))
ax = sns.barplot(x=list(metrics.keys()), y=list(metrics.values()), palette=palette)

for p in ax.patches:
    ax.annotate(format(p.get_height(), '.2f') + '%',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, 9), textcoords='offset points', weight='bold')

plt.title('Final Evaluation Metrics', fontsize=16, pad=20)
plt.ylabel('Score (%)')
plt.ylim(0, 115)
plt.tight_layout()
plt.show()

#### **📊 Naive Bayes Sentiment Classification Metrics Guide**

In this project, we use a standard set of classification metrics to evaluate the quality of our Arabic sentiment classifier.

---

##### 1. Accuracy
* **Description:** The percentage of all reviews classified correctly.
* **Range:** 0% to 100%.
* **Interpretation:**
    * **< 70%:** Weak; the model is struggling with Arabic text patterns.
    * **70% - 85%:** **(The Success Zone)** Good result for a Bag-of-Words + Naive Bayes baseline.
    * **> 85%:** Excellent; strong signal extraction from the preprocessing pipeline.

---

##### 2. Precision
* **Description:** Of all reviews predicted as Positive, how many were actually Positive?
* **Why it matters:** Tells us how trustworthy the model's "Positive" predictions are.
* **Interpretation:** Scores above **75%** indicate the model is not falsely labeling negative reviews as positive.

---

##### 3. Recall
* **Description:** Of all actual Positive reviews, how many did the model catch?
* **Why it matters:** Tells us how well the model avoids missing real positive sentiment.
* **Interpretation:** A balanced Precision/Recall is ideal; a large gap suggests class imbalance issues.

---

##### 4. F1-Score
* **Description:** The harmonic mean of Precision and Recall — a single balanced metric.
* **Range:** 0% to 100%.
* **Interpretation:** Higher F1 means the model handles both false positives and false negatives well.

---

##### 📝 Summary Table: Are your results good?

| Metric | Weak Result | Good (Lab Project) | Excellent |
| :--- | :--- | :--- | :--- |
| **Accuracy** | < 70% | 75% - 85% | 88%+ |
| **Precision** | < 65% | 70% - 82% | 85%+ |
| **Recall** | < 65% | 70% - 82% | 85%+ |
| **F1-Score** | < 65% | 72% - 83% | 86%+ |

---

##### 💡 Important Notes for Discussion:
1.  **Stemming Impact:** ISRIStemmer normalizes Arabic morphology aggressively. If accuracy drops, try skipping stemming and compare.
2.  **Stopwords:** Arabic stopwords vary by dialect. If reviews are dialectal (Egyptian, Gulf, etc.), the NLTK stopword list may be incomplete.
3.  **Error Analysis:** Always inspect wrong predictions — look for very short reviews or reviews with sarcasm, which are hard for Bag-of-Words models.


## 6️⃣ Testing

In [ ]:
def predict_sentiment(review_text):
    cleaned = full_preprocess(review_text)
    vector = vectorizer.transform([cleaned])
    prediction = model.predict(vector)[0]
    proba = model.predict_proba(vector)[0]
    classes = model.classes_
    confidence = {cls: f"{prob*100:.1f}%" for cls, prob in zip(classes, proba)}
    print(f"Review    : {review_text}")
    print(f"Prediction: {prediction}")
    print(f"Confidence: {confidence}")
    print("-" * 50)

In [ ]:
predict_sentiment("هذا التطبيق ممتاز وسهل الاستخدام")

In [ ]:
predict_sentiment("المنتج سيء جداً ولا أنصح بشرائه")

In [ ]:
predict_sentiment("الفيلم كان رائعاً وممتعاً للغاية")

In [ ]:
predict_sentiment("خدمة العملاء بطيئة ومحبطة")

## **Thank You** 🎀💌💓